In [1]:
import os
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt  <-- Toujours désactivé
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier # <-- Changement ici
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, brier_score_loss, roc_curve

# ============================================================
# CONFIGURATION
# ============================================================
DATA_PATH = "../datasets/schaefcomb_Wang2023Simple_dfschizo.tsv"

CONF_METHOD = "regression" 
INCLUDE_CONFOUNDS_IN_MODEL = False 
N_PCA_COMPONENTS = 100

# ============================================================
# 1) CHARGEMENT ET PRÉPARATION
# ============================================================
print(f"--- Chargement : {DATA_PATH} ---")
df_all = pd.read_csv(DATA_PATH, sep="\t")

mask_loso = (
    df_all["site_id"].isin(["ds000030", "ds_cobre", "ds004302"])
    & df_all["diagnosis"].isin(["CONTROL", "SCHZ"])
)
df_loso = df_all.loc[mask_loso].copy()

# Variables cliniques
df_loso["age"] = df_loso["age"].astype(float)
df_loso["gender_num"] = (df_loso["gender"] == "M").astype(float)

# Gestion mean_fd
if "mean_fd" not in df_loso.columns:
    df_loso["mean_fd"] = 0.0
else:
    df_loso["mean_fd"] = df_loso["mean_fd"].astype(float)
    if df_loso["mean_fd"].isna().any():
        df_loso["mean_fd"] = df_loso["mean_fd"].fillna(df_loso["mean_fd"].median())

# Cible et Groupes
y_loso_all = (df_loso["diagnosis"] == "SCHZ").astype(int)
groups_loso = df_loso["site_id"].values 

# ============================================================
# 2) CONNECTOMES
# ============================================================
corr_cols_all = [c for c in df_loso.columns if c.startswith("corr_")]
X_temp = df_loso[corr_cols_all]
corr_cols = X_temp.columns[X_temp.notna().any()].tolist()

imputer = SimpleImputer(strategy="mean")
X_loso_imp = pd.DataFrame(
    imputer.fit_transform(df_loso[corr_cols]), 
    index=df_loso.index, columns=corr_cols
)

# ============================================================
# 3) FONCTIONS UTILES (CORRECTION + MODÈLE PCA-RF)
# ============================================================

def residualize_confounds(X_train, X_test, conf_train, conf_test):
    """Régression linéaire : Age + Sexe + FD"""
    C_train = np.column_stack([np.ones(len(conf_train)), conf_train.values])
    C_test  = np.column_stack([np.ones(len(conf_test)),  conf_test.values])
    B = np.linalg.pinv(C_train).dot(X_train.values)
    return (pd.DataFrame(X_train.values - C_train.dot(B), index=X_train.index, columns=X_train.columns),
            pd.DataFrame(X_test.values - C_test.dot(B), index=X_test.index, columns=X_test.columns))

def get_model_pca():
    """Pipeline : Scaler -> PCA (100) -> Random Forest (GridSearch)"""
    pca = PCA(n_components=N_PCA_COMPONENTS, random_state=42)
    
    # Random Forest
    rf = RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
    
    # Grille de recherche simplifiée pour éviter un temps de calcul trop long
    clf = GridSearchCV(
        estimator=rf,
        param_grid={
            'n_estimators': [100, 200],      # Nombre d'arbres
            'max_depth': [None, 10, 20],     # Profondeur max
            # 'min_samples_leaf': [1, 4]     # Optionnel : réduire l'overfitting
        },
        cv=5,
        scoring='roc_auc',
        n_jobs=-1
    )
    return Pipeline([("scaler", StandardScaler()), ("pca", pca), ("clf", clf)])

# ============================================================
# 4) BOUCLE LOSO (DONNÉES RÉELLES)
# ============================================================
print("\n" + "="*60)
print(f"DÉBUT LOSO (REAL DATA) | PCA ({N_PCA_COMPONENTS}) + Random Forest")
print("="*60)

logo = LeaveOneGroupOut()
loso_results = []
feature_weights_list = []

for i, (train_idx, test_idx) in enumerate(logo.split(X_loso_imp, y_loso_all, groups=groups_loso)):
    site_test_name = groups_loso[test_idx][0]
    print(f"\n🔹 SITE TEST : {site_test_name}")
    
    # 1. Split
    X_tr, X_te = X_loso_imp.iloc[train_idx], X_loso_imp.iloc[test_idx]
    y_tr, y_te = y_loso_all.iloc[train_idx], y_loso_all.iloc[test_idx]
    
    conf_vars = ["age", "gender_num", "mean_fd"]
    c_tr, c_te = df_loso.iloc[train_idx][conf_vars], df_loso.iloc[test_idx][conf_vars]

    # 2. Correction
    if CONF_METHOD == "regression":
        X_tr_cl, X_te_cl = residualize_confounds(X_tr, X_te, c_tr, c_te)
    else:
        X_tr_cl, X_te_cl = X_tr, X_te

    # 3. Fit
    pipeline = get_model_pca()
    pipeline.fit(X_tr_cl, y_tr)
    
    # 4. Metrics
    clf_obj = pipeline.named_steps['clf']
    
    # Récupération des meilleurs params
    best_est = clf_obj.best_params_['n_estimators']
    best_depth = clf_obj.best_params_['max_depth']
    auc_int = clf_obj.best_score_
    
    # Prédictions
    y_prob = pipeline.predict_proba(X_te_cl)[:, 1]
    auc_ext = roc_auc_score(y_te, y_prob)
    brier = brier_score_loss(y_te, y_prob)
    
    print(f"   Params: est={best_est}, depth={best_depth} | AUC CV: {auc_int:.3f} | AUC Ext: {auc_ext:.3f}")

    loso_results.append({
        "site": site_test_name, "auc_int": auc_int, "auc_ext": auc_ext, 
        "delta": auc_int - auc_ext, "brier": brier
    })

    # 5. Extraction Importance des Features (Back-projection RF)
    # a. Importance donnée aux composantes PCA (somme = 1)
    importances_pca = clf_obj.best_estimator_.feature_importances_ 
    
    # b. Loadings de la PCA (n_components, n_features)
    pca_obj = pipeline.named_steps['pca']
    components = pca_obj.components_ 
    
    # c. Projection : Importance Originale = Importance_PCA * abs(Loadings_PCA)
    # On utilise abs() car l'importance RF est sans signe (magnitude d'influence).
    reconstructed_weights = np.dot(importances_pca, np.abs(components)).flatten()
    
    feature_weights_list.append(pd.DataFrame({
        "feature": X_tr_cl.columns, 
        "weight": reconstructed_weights, # Ici, "weight" = Importance relative (toujours positif)
        "test_site": site_test_name
    }))

# Tableau Résumé
df_res = pd.DataFrame(loso_results)
print("\n--- RÉSULTATS REAL DATA (PCA-RF) ---")
print(df_res[["site", "auc_int", "auc_ext", "delta", "brier"]].round(3))
print(f"Moyenne AUC Externe : {df_res['auc_ext'].mean():.3f}")

# ============================================================
# 5) SANITY CHECK (RANDOM)
# ============================================================
print("\n" + "="*60)
print(f"DÉBUT SANITY CHECK")
print("="*60)

np.random.seed(99)
y_rand = pd.Series(np.random.permutation(y_loso_all), index=df_loso.index)
rand_results = []

for i, (train_idx, test_idx) in enumerate(logo.split(X_loso_imp, y_rand, groups=groups_loso)):
    site_test_name = groups_loso[test_idx][0]
    X_tr, X_te = X_loso_imp.iloc[train_idx], X_loso_imp.iloc[test_idx]
    y_tr, y_te = y_rand.iloc[train_idx], y_rand.iloc[test_idx]
    c_tr, c_te = df_loso.iloc[train_idx][conf_vars], df_loso.iloc[test_idx][conf_vars]
    
    if CONF_METHOD == "regression":
        X_tr_cl, X_te_cl = residualize_confounds(X_tr, X_te, c_tr, c_te)
    else:
        X_tr_cl, X_te_cl = X_tr, X_te
        
    pipeline = get_model_pca()
    pipeline.fit(X_tr_cl, y_tr)
    y_prob = pipeline.predict_proba(X_te_cl)[:, 1]
    
    auc_r = roc_auc_score(y_te, y_prob)
    print(f"   Site {site_test_name} | AUC Random: {auc_r:.3f}")
    rand_results.append({"site": site_test_name, "auc_random": auc_r})

print("\n--- RÉSULTATS RANDOM CHECK ---")
df_rand = pd.DataFrame(rand_results)
print(f"Moyenne AUC Random : {df_rand['auc_random'].mean():.3f}")

# ============================================================
# 6) VISUALISATION TEXTUELLE UNIQUEMENT
# ============================================================
print("\n" + "="*60)
print("VISUALISATION TEXTUELLE (Top Importances)")
print("="*60)

if feature_weights_list:
    df_w = pd.concat(feature_weights_list, ignore_index=True)
    df_cons = df_w.groupby("feature").agg({"weight": "mean"})
    # Le poids est déjà positif (importance), on trie directement
    df_cons = df_cons.sort_values(by="weight", ascending=False)
    
    print("--- TOP 10 CONNEXIONS (Importance RF reconstruite) ---")
    print(df_cons.head(10))
else:
    print("Aucun poids récupéré.")

--- Chargement : ../datasets/schaefcomb_Wang2023Simple_dfschizo.tsv ---


/tmp/ipykernel_3520246/1640011442.py:26: DtypeWarning: Columns (93968) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all = pd.read_csv(DATA_PATH, sep="\t")



DÉBUT LOSO (REAL DATA) | PCA (100) + Random Forest

🔹 SITE TEST : ds000030
   Params: est=200, depth=None | AUC CV: 0.667 | AUC Ext: 0.654

🔹 SITE TEST : ds004302
   Params: est=200, depth=None | AUC CV: 0.682 | AUC Ext: 0.720

🔹 SITE TEST : ds_cobre
   Params: est=200, depth=10 | AUC CV: 0.737 | AUC Ext: 0.592

--- RÉSULTATS REAL DATA (PCA-RF) ---
       site  auc_int  auc_ext  delta  brier
0  ds000030    0.667    0.654  0.013  0.242
1  ds004302    0.682    0.720 -0.037  0.306
2  ds_cobre    0.737    0.592  0.144  0.244
Moyenne AUC Externe : 0.655

DÉBUT SANITY CHECK
   Site ds000030 | AUC Random: 0.459
   Site ds004302 | AUC Random: 0.467
   Site ds_cobre | AUC Random: 0.440

--- RÉSULTATS RANDOM CHECK ---
Moyenne AUC Random : 0.456

VISUALISATION TEXTUELLE (Top Importances)
--- TOP 10 CONNEXIONS (Importance RF reconstruite) ---
                weight
feature               
corr_296_317  0.003279
corr_317_383  0.003242
corr_183_317  0.003219
corr_201_313  0.003205
corr_244_317  0.00